# WebNavi: implementação da DSL em Python

Este notebook implementa uma versão funcional da DSL **WebNavi**, uma linguagem declarativa em português para modelar fluxos de navegação web, páginas, elementos, transições, dados de teste, invariantes e fluxos de teste.

A implementação lê um programa em WebNavi, constrói uma representação interna, executa análise estática simples e gera um esboço de teste Cypress.

In [1]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
import re

@dataclass
class Elemento:
    nome: str
    tipo: str
    modificadores: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Pagina:
    nome: str
    rota: str = ""
    elementos: Dict[str, Elemento] = field(default_factory=dict)

@dataclass
class Transicao:
    origem: str
    destino: str
    elemento: str
    acao: str
    condicao: Optional[str] = None
    efeito: Optional[str] = None

@dataclass
class Invariante:
    nome: str
    regra: str

@dataclass
class Fluxo:
    nome: str
    inicio: str
    passos: List[str]
    fim: str

@dataclass
class ProgramaWebNavi:
    site: str = ""
    paginas: Dict[str, Pagina] = field(default_factory=dict)
    inicio: Optional[str] = None
    finais: List[str] = field(default_factory=list)
    transicoes: List[Transicao] = field(default_factory=list)
    invariantes: List[Invariante] = field(default_factory=list)
    dados: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    fluxos: Dict[str, Fluxo] = field(default_factory=dict)

## Parser

O parser é linha a linha e reconhece os blocos principais da linguagem: `site`, `pagina`, `inicio`, `final`, `transicao`, `invariante`, `dados` e `fluxo`.

In [2]:
def remover_comentario(linha: str) -> str:
    # Remove comentários iniciados por # apenas quando o # está fora de aspas.
    em_aspas = False
    resultado = []
    for ch in linha:
        if ch == '\"':
            em_aspas = not em_aspas
        if ch == '#' and not em_aspas:
            break
        resultado.append(ch)
    return ''.join(resultado).rstrip()

def limpa_linhas(codigo: str) -> List[str]:
    linhas = []
    for linha in codigo.splitlines():
        sem_comentario = remover_comentario(linha)
        if sem_comentario.strip():
            linhas.append(sem_comentario)
    return linhas

def parse_valor(token: str) -> Any:
    token = token.strip()
    if token.startswith('"') and token.endswith('"'):
        return token[1:-1]
    if token == 'verdadeiro':
        return True
    if token == 'falso':
        return False
    if token.isdigit():
        return int(token)
    return token

def tokenizar_linha(linha: str) -> List[str]:
    return re.findall(r'"[^"]*"|\S+', linha)

def parse_modificadores(partes: List[str]) -> Dict[str, Any]:
    mods = {}
    i = 0
    while i < len(partes):
        p = partes[i]
        if p == 'obrigatorio':
            mods['obrigatorio'] = True
            i += 1
        elif p in ('formato', 'minimo', 'maximo', 'texto', 'seletor') and i + 1 < len(partes):
            mods[p] = parse_valor(partes[i + 1])
            i += 2
        elif p == 'visivel' and i + 2 < len(partes) and partes[i + 1] == 'quando':
            mods['visivel_quando'] = partes[i + 2]
            i += 3
        else:
            mods.setdefault('_extras', []).append(p)
            i += 1
    return mods

def parse_webnavi(codigo: str) -> ProgramaWebNavi:
    linhas = limpa_linhas(codigo)
    prog = ProgramaWebNavi()
    i = 0
    while i < len(linhas):
        linha = linhas[i].strip()
        if linha.startswith('site '):
            prog.site = linha.split(maxsplit=1)[1]
            i += 1
        elif linha.startswith('pagina '):
            nome = linha.split(maxsplit=1)[1]
            pagina = Pagina(nome=nome)
            i += 1
            while i < len(linhas) and linhas[i].strip() != 'fim':
                atual = linhas[i].strip()
                if atual.startswith('rota '):
                    pagina.rota = atual.split(maxsplit=1)[1]
                elif atual.startswith('elemento '):
                    partes = tokenizar_linha(atual)
                    nome_elem, tipo = partes[1], partes[2]
                    pagina.elementos[nome_elem] = Elemento(nome_elem, tipo, parse_modificadores(partes[3:]))
                i += 1
            prog.paginas[nome] = pagina
            i += 1
        elif linha.startswith('inicio em '):
            prog.inicio = linha.replace('inicio em ', '', 1).strip()
            i += 1
        elif linha.startswith('final em '):
            prog.finais.extend(linha.replace('final em ', '', 1).replace(',', ' ').split())
            i += 1
        elif linha.startswith('transicao de '):
            m = re.match(r'transicao de (\w+) para (\w+)', linha)
            if not m:
                raise SyntaxError(f'Transição inválida: {linha}')
            origem, destino = m.group(1), m.group(2)
            elemento = acao = None
            condicao = efeito = None
            i += 1
            while i < len(linhas) and linhas[i].strip() != 'fim':
                atual = linhas[i].strip()
                if atual.startswith('via '):
                    partes = atual.split()
                    elemento, acao = partes[1], partes[2]
                elif atual.startswith('somente se '):
                    condicao = atual.replace('somente se ', '', 1)
                elif atual.startswith('entao '):
                    efeito = atual.replace('entao ', '', 1)
                i += 1
            prog.transicoes.append(Transicao(origem, destino, elemento, acao, condicao, efeito))
            i += 1
        elif linha.startswith('invariante '):
            nome = linha.replace('invariante ', '', 1).strip()
            regra = ''
            i += 1
            while i < len(linhas) and linhas[i].strip() != 'fim':
                regra += linhas[i].strip() + ' '
                i += 1
            prog.invariantes.append(Invariante(nome, regra.strip()))
            i += 1
        elif linha.startswith('dados '):
            nome = linha.split(maxsplit=1)[1]
            campos = {}
            i += 1
            while i < len(linhas) and linhas[i].strip() != 'fim':
                partes = tokenizar_linha(linhas[i].strip())
                campos[partes[0]] = parse_valor(' '.join(partes[1:]))
                i += 1
            prog.dados[nome] = campos
            i += 1
        elif linha.startswith('fluxo '):
            nome = linha.replace('fluxo ', '', 1).strip().replace(' ', '_')
            inicio = fim = None
            passos = []
            i += 1
            while i < len(linhas) and linhas[i].strip() != 'fim':
                atual = linhas[i].strip()
                if atual.startswith('comecar em '):
                    inicio = atual.replace('comecar em ', '', 1).strip()
                elif atual.startswith('terminar em '):
                    fim = atual.replace('terminar em ', '', 1).strip()
                elif atual.startswith('passo '):
                    passos.append(atual.replace('passo ', '', 1).strip())
                i += 1
            prog.fluxos[nome] = Fluxo(nome, inicio, passos, fim)
            i += 1
        else:
            raise SyntaxError(f'Linha não reconhecida: {linha}')
    return prog

## Análise estática

In [3]:
def analisar_programa(prog: ProgramaWebNavi) -> Dict[str, List[str]]:
    erros, avisos, oks = [], [], []
    if not prog.site:
        erros.append('Nenhum site foi declarado.')
    if not prog.inicio:
        erros.append('Nenhum estado inicial foi declarado.')
    elif prog.inicio not in prog.paginas:
        erros.append(f'Página inicial não declarada: {prog.inicio}')
    if not prog.transicoes:
        erros.append('Nenhuma transição foi declarada.')
    for final in prog.finais:
        if final not in prog.paginas:
            erros.append(f'Página final não declarada: {final}')
    for t in prog.transicoes:
        if t.origem not in prog.paginas:
            erros.append(f'Transição com origem inexistente: {t.origem}')
        if t.destino not in prog.paginas:
            erros.append(f'Transição com destino inexistente: {t.destino}')
        if t.origem in prog.paginas and t.elemento not in prog.paginas[t.origem].elementos:
            avisos.append(f'Elemento {t.elemento} não declarado na página de origem {t.origem}.')
    alcancaveis = set()
    if prog.inicio in prog.paginas:
        fila = [prog.inicio]
        while fila:
            atual = fila.pop(0)
            if atual in alcancaveis:
                continue
            alcancaveis.add(atual)
            for t in prog.transicoes:
                if t.origem == atual and t.destino not in alcancaveis:
                    fila.append(t.destino)
    for p in prog.paginas:
        if p in alcancaveis:
            oks.append(f'Página alcançável: {p}')
        else:
            avisos.append(f'Página não alcançável a partir de {prog.inicio}: {p}')
    for nome, fluxo in prog.fluxos.items():
        if fluxo.inicio not in prog.paginas:
            erros.append(f'Fluxo {nome} começa em página inexistente: {fluxo.inicio}')
        if fluxo.fim not in prog.paginas:
            erros.append(f'Fluxo {nome} termina em página inexistente: {fluxo.fim}')
        oks.append(f'Fluxo {nome}: {len(fluxo.passos)} passos')
    return {'ok': oks, 'avisos': avisos, 'erros': erros}

def imprimir_relatorio(prog: ProgramaWebNavi):
    analise = analisar_programa(prog)
    print(f'WebNavi — Análise Estática de {prog.site}.webnavi')
    print('=' * 55)
    print(f'Páginas declaradas: {len(prog.paginas)}')
    print(', '.join(prog.paginas.keys()))
    print('\n[OK]')
    for item in analise['ok']:
        print(' -', item)
    print('\n[AVISOS]')
    for item in analise['avisos'] or ['Nenhum aviso.']:
        print(' -', item)
    print('\n[ERROS]')
    for item in analise['erros'] or ['Nenhum erro.']:
        print(' -', item)

## Geração de testes Cypress

In [4]:
def seletor_elemento(prog: ProgramaWebNavi, nome_elemento: str) -> str:
    for pagina in prog.paginas.values():
        if nome_elemento in pagina.elementos:
            elem = pagina.elementos[nome_elemento]
            return elem.modificadores.get('seletor', f'[data-webnavi="{nome_elemento}"]')
    return f'[data-webnavi="{nome_elemento}"]'

def rota_pagina(prog: ProgramaWebNavi, nome_pagina: str) -> str:
    return prog.paginas.get(nome_pagina, Pagina(nome_pagina, f'/{nome_pagina}')).rota or f'/{nome_pagina}'

def resolver_referencia_dados(prog: ProgramaWebNavi, ref: str) -> Any:
    ref = ref.strip()
    if ref.startswith('"') and ref.endswith('"'):
        return ref[1:-1]
    if '.' in ref:
        conjunto, campo = ref.split('.', 1)
        return prog.dados.get(conjunto, {}).get(campo, ref)
    return ref

def gerar_cypress(prog: ProgramaWebNavi, nome_fluxo: str) -> str:
    fluxo = prog.fluxos[nome_fluxo]
    linhas = [
        f"// GERADO AUTOMATICAMENTE de: {prog.site}.webnavi :: fluxo {fluxo.nome.replace('_', ' ')}",
        f"describe('fluxo: {fluxo.nome.replace('_', ' ')}', () => {{",
        "  it('deve executar o fluxo declarado', () => {",
        f"    cy.visit('{rota_pagina(prog, fluxo.inicio)}');"
    ]
    for passo in fluxo.passos:
        acao = passo.split()[0]
        if acao == 'clicar':
            elem = passo.split()[1]
            linhas.append(f"    cy.get('{seletor_elemento(prog, elem)}').click();")
        elif acao == 'preencher':
            m = re.match(r'preencher (\w+) com (.+?)(?:\s+aguardar\s+.+)?$', passo)
            if m:
                elem, ref = m.group(1), m.group(2).strip()
                valor = resolver_referencia_dados(prog, ref)
                linhas.append(f"    cy.get('{seletor_elemento(prog, elem)}').type('{valor}');")
        elif acao == 'navegar':
            pagina = passo.split()[1]
            linhas.append(f"    cy.visit('{rota_pagina(prog, pagina)}');")
        if 'aguardar pagina ' in passo:
            destino = passo.split('aguardar pagina ', 1)[1].split()[0]
            linhas.append(f"    cy.url().should('include', '{rota_pagina(prog, destino)}');")
        elif 'aguardar ' in passo and ' visivel' in passo:
            elem = passo.split('aguardar ', 1)[1].replace(' visivel', '').strip()
            linhas.append(f"    cy.get('{seletor_elemento(prog, elem)}').should('be.visible');")
    linhas += ["  });", "});"]
    return '\n'.join(linhas)

## Exemplo executável

In [5]:
EXEMPLO_LOJA = """
site loja

pagina inicio
  rota /
  elemento botao_login botao texto "Entrar" seletor "#login"
fim

pagina login
  rota /login
  elemento campo_email entrada obrigatorio formato email seletor "#email"
  elemento campo_senha entrada obrigatorio formato senha minimo 8 seletor "#senha"
  elemento botao_entrar botao texto "Entrar" seletor "#btn-entrar"
  elemento aviso_erro aviso visivel quando autenticacao_falhou seletor ".erro"
fim

pagina catalogo
  rota /produtos
  elemento botao_adicionar botao texto "Adicionar" seletor "#adicionar"
fim

pagina carrinho
  rota /carrinho
  elemento botao_finalizar botao texto "Finalizar" seletor "#finalizar"
fim

pagina checkout
  rota /checkout
  elemento campo_cartao entrada obrigatorio minimo 16 maximo 16 seletor "#cartao"
  elemento campo_cvv entrada obrigatorio minimo 3 maximo 4 seletor "#cvv"
  elemento botao_pagar botao texto "Pagar" seletor "#pagar"
fim

pagina confirmacao
  rota /confirmacao
  elemento numero_pedido entrada seletor "#numero-pedido"
fim

inicio em inicio
final em confirmacao

transicao de inicio para login
  via botao_login clicado
fim

transicao de login para catalogo
  via botao_entrar clicado
  somente se campo_email preenchido e campo_senha preenchido
  entao sessao.autenticado recebe verdadeiro
fim

transicao de catalogo para carrinho
  via botao_adicionar clicado
fim

transicao de carrinho para checkout
  via botao_finalizar clicado
  somente se sessao.autenticado igual a verdadeiro
fim

transicao de checkout para confirmacao
  via botao_pagar clicado
  somente se campo_cartao preenchido e campo_cvv preenchido
fim

invariante checkout exige autenticacao
  sempre que estiver em checkout entao sessao.autenticado igual a verdadeiro
fim

dados usuario_valido
  email "usuario@teste.com"
  senha "Senha@123"
fim

dados cartao_valido
  numero "4111111111111111"
  cvv "123"
fim

fluxo compra completa
  comecar em inicio
  passo clicar botao_login aguardar pagina login
  passo preencher campo_email com usuario_valido.email
  passo preencher campo_senha com usuario_valido.senha
  passo clicar botao_entrar aguardar pagina catalogo
  passo clicar botao_adicionar aguardar pagina carrinho
  passo clicar botao_finalizar aguardar pagina checkout
  passo preencher campo_cartao com cartao_valido.numero
  passo preencher campo_cvv com cartao_valido.cvv
  passo clicar botao_pagar aguardar pagina confirmacao
  terminar em confirmacao
fim
"""

programa = parse_webnavi(EXEMPLO_LOJA)
print(programa)

ProgramaWebNavi(site='loja', paginas={'inicio': Pagina(nome='inicio', rota='/', elementos={'botao_login': Elemento(nome='botao_login', tipo='botao', modificadores={'texto': 'Entrar', 'seletor': '#login'})}), 'login': Pagina(nome='login', rota='/login', elementos={'campo_email': Elemento(nome='campo_email', tipo='entrada', modificadores={'obrigatorio': True, 'formato': 'email', 'seletor': '#email'}), 'campo_senha': Elemento(nome='campo_senha', tipo='entrada', modificadores={'obrigatorio': True, 'formato': 'senha', 'minimo': 8, 'seletor': '#senha'}), 'botao_entrar': Elemento(nome='botao_entrar', tipo='botao', modificadores={'texto': 'Entrar', 'seletor': '#btn-entrar'}), 'aviso_erro': Elemento(nome='aviso_erro', tipo='aviso', modificadores={'visivel_quando': 'autenticacao_falhou', 'seletor': '.erro'})}), 'catalogo': Pagina(nome='catalogo', rota='/produtos', elementos={'botao_adicionar': Elemento(nome='botao_adicionar', tipo='botao', modificadores={'texto': 'Adicionar', 'seletor': '#adicio

In [6]:
imprimir_relatorio(programa)

WebNavi — Análise Estática de loja.webnavi
Páginas declaradas: 6
inicio, login, catalogo, carrinho, checkout, confirmacao

[OK]
 - Página alcançável: inicio
 - Página alcançável: login
 - Página alcançável: catalogo
 - Página alcançável: carrinho
 - Página alcançável: checkout
 - Página alcançável: confirmacao
 - Fluxo compra_completa: 9 passos

[AVISOS]
 - Nenhum aviso.

[ERROS]
 - Nenhum erro.


In [7]:
print(gerar_cypress(programa, 'compra_completa'))

// GERADO AUTOMATICAMENTE de: loja.webnavi :: fluxo compra completa
describe('fluxo: compra completa', () => {
  it('deve executar o fluxo declarado', () => {
    cy.visit('/');
    cy.get('#login').click();
    cy.url().should('include', '/login');
    cy.get('#email').type('usuario@teste.com');
    cy.get('#senha').type('Senha@123');
    cy.get('#btn-entrar').click();
    cy.url().should('include', '/produtos');
    cy.get('#adicionar').click();
    cy.url().should('include', '/carrinho');
    cy.get('#finalizar').click();
    cy.url().should('include', '/checkout');
    cy.get('#cartao').type('4111111111111111');
    cy.get('#cvv').type('123');
    cy.get('#pagar').click();
    cy.url().should('include', '/confirmacao');
  });
});


## Próximos passos possíveis

Esta versão mostra a viabilidade da DSL. Em uma versão futura, o parser linha a linha poderia ser substituído por uma gramática formal usando ANTLR ou Lark, e a análise semântica poderia detectar condições contraditórias, estados presos, cobertura de invariantes e geração completa de testes end-to-end.